# ⭐ Day 72: Reinforcement Learning Fundamentals
# Q-Learning and Basic Decision Making
### 🚀 Day 72 of 369-day Python & AI Learning Path
---
*"The best way to predict the future is to create it." — Peter Drucker*


## Welcome to the World of Reinforcement Learning! 🤖

Welcome to Day 72 of your Python & AI Learning Path! Today marks a pivotal moment in your journey — we are stepping into the fascinating and powerful world of **Reinforcement Learning (RL)**. Unlike the supervised and unsupervised learning paradigms you've explored so far, RL represents a fundamentally different approach to machine intelligence. Here, we don't feed our models pre-labeled datasets. Instead, we create **agents** that learn by doing, exploring, and experiencing the consequences of their actions.

Imagine teaching a robot to walk. You could try to manually program every joint movement, but that would be impossibly complex. Alternatively, you could let the robot try, fall, get back up, and slowly discover which movements lead to stable walking. This is the essence of Reinforcement Learning — learning through **trial and error**, guided by the pursuit of rewards and the avoidance of penalties.

Today, we will build the foundational understanding you need by diving into **Q-Learning**, one of the most elegant and influential algorithms in the history of AI. Q-Learning allows an agent to learn optimal strategies (policies) for decision-making without any prior knowledge of the environment's dynamics. By the end of this notebook, you will have built your very own RL agent from scratch and watched it master simple environments through pure experience.

So buckle up! We're about to make your code come alive with curiosity, exploration, and intelligent decision-making. Let's turn your Python skills into the building blocks of artificial intelligence that can learn, adapt, and improve. 🎮✨


## 📋 Table of Contents

1. [Introduction to Reinforcement Learning](#1.-Introduction-to-Reinforcement-Learning)
2. [Key Concepts: MDP, Policy, Value Function](#2.-Key-Concepts:-Markov-Decision-Process,-Policy,-Value-Function)
3. [The Exploration vs Exploitation Dilemma](#3.-The-Exploration-vs-Exploitation-Dilemma)
4. [Q-Learning Algorithm Explained](#4.-Q-Learning-Algorithm-Explained)
5. [Implementing Q-Learning from Scratch on a Simple Grid World](#5.-Implementing-Q-Learning-from-Scratch-on-a-Simple-Grid-World)
6. [Training Loop, Epsilon-Greedy Policy, and Convergence](#6.-Training-Loop,-Epsilon-Greedy-Policy,-and-Convergence)
7. [Visualizing the Learned Policy and Q-Table](#7.-Visualizing-the-Learned-Policy-and-Q-Table)
8. [Applying Q-Learning to a Classic Environment](#8.-Applying-Q-Learning-to-a-Classic-Environment)
9. [Limitations and the Road to Deep RL](#9.-Limitations-of-Tabular-Q-Learning-and-the-Road-to-Deep-RL)
10. [Pros, Cons, and Real-World Applications](#10.-Pros,-Cons,-and-Real-World-Applications)
11. [🛠️ Hands-On Exercises](#Hands-On-Exercises)
12. [Solutions](#Solutions)


## 1. Introduction to Reinforcement Learning 🎯

Reinforcement Learning is a type of machine learning where an **agent** learns to make decisions by interacting with an **environment**. The agent observes the current **state**, takes an **action**, and receives a **reward** (or penalty) in return. The goal is to maximize the total cumulative reward over time.

### Core Components of RL:

| Component | Symbol | Description |
|-----------|--------|-------------|
| **Agent** | — | The learner and decision-maker |
| **Environment** | — | The world the agent interacts with |
| **State** | $s$ | A representation of the current situation |
| **Action** | $a$ | A choice made by the agent |
| **Reward** | $r$ | Feedback from the environment (scalar value) |
| **Policy** | $\pi$ | The strategy the agent uses to choose actions |
| **Episode** | — | A sequence of states, actions, and rewards from start to end |

### The RL Loop:
```
┌─────────┐     Action (a)     ┌─────────────┐
│  Agent  │ ─────────────────> │ Environment │
│         │                    │             │
│         │ <───────────────── │             │
└─────────┘   New State (s')   └─────────────┘
       ^         & Reward (r)
       │
       └────────────────────────┘
```

**Key Insight:** The agent doesn't know the rules of the game upfront. It must discover them through interaction — much like how humans learn to play video games or ride a bicycle!


In [ ]:
# Let's set up our environment and import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import random
import time
from IPython.display import clear_output

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

print("✅ Libraries imported successfully!")
print("🐍 NumPy version:", np.__version__)
print("📊 Ready to build our first RL agent!")


## 2. Key Concepts: Markov Decision Process, Policy, Value Function 📐

### Markov Decision Process (MDP)
An MDP provides the mathematical framework for modeling decision-making in situations where outcomes are partly random and partly under the control of a decision-maker. An MDP is defined by:
- **States ($S$):** All possible situations
- **Actions ($A$):** All possible choices
- **Transition Probabilities ($P(s'|s,a)$):** Probability of moving to state $s'$ given action $a$ in state $s$
- **Reward Function ($R(s,a)$):** Expected reward for taking action $a$ in state $s$
- **Discount Factor ($\gamma$):** How much we value future rewards (0 to 1)

### The Markov Property
*"The future is independent of the past given the present."*
$$P(s_{t+1} | s_t, a_t) = P(s_{t+1} | s_1, a_1, ..., s_t, a_t)$$

### Policy ($\pi$)
A policy is a mapping from states to actions. It defines the agent's behavior:
- **Deterministic Policy:** $\pi(s) = a$ (always choose action $a$ in state $s$)
- **Stochastic Policy:** $\pi(a|s)$ = probability of choosing action $a$ in state $s$

### Value Functions
- **State-Value Function $V(s)$:** Expected return starting from state $s$ and following policy $\pi$
- **Action-Value Function $Q(s,a)$:** Expected return starting from state $s$, taking action $a$, then following policy $\pi$

The $Q$-function is what Q-Learning aims to learn! 💡


In [ ]:
# Let's create a simple visualization of the RL components
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw components
components = [
    (2, 7, 'Agent\n🤖', '#FF6B6B'),
    (8, 7, 'Environment\n🌍', '#4ECDC4'),
    (5, 3, 'Reward\n🏆', '#FFE66D'),
    (5, 7, 'State\n📍', '#95E1D3')
]

for x, y, text, color in components:
    circle = plt.Circle((x, y), 1.2, color=color, alpha=0.7)
    ax.add_patch(circle)
    ax.text(x, y, text, ha='center', va='center', fontsize=11, fontweight='bold')

# Draw arrows
ax.annotate('', xy=(6.5, 7), xytext=(3.5, 7),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(5, 7.8, 'Action (a)', ha='center', fontsize=10, style='italic')

ax.annotate('', xy=(3.5, 5.5), xytext=(6.5, 5.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(5, 5.2, 'New State + Reward', ha='center', fontsize=10, style='italic')

ax.set_title('The Reinforcement Learning Loop', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("💡 The agent learns by continuously interacting with the environment!")


## 3. The Exploration vs Exploitation Dilemma ⚖️

One of the most fundamental challenges in RL is balancing **exploration** (trying new things) with **exploitation** (using what you already know works well).

### The Restaurant Analogy 🍕
Imagine you're in a new city with 10 restaurants:
- **Exploitation:** Go to the pizza place you loved yesterday (guaranteed good meal)
- **Exploration:** Try the new sushi place (might be amazing, might be terrible)

If you only exploit, you might miss the best restaurant in town. If you only explore, you'll never enjoy a consistently good meal. The optimal strategy is to explore early and gradually shift to exploitation as you learn.

### Epsilon-Greedy Strategy
The most common approach is the **epsilon-greedy** policy:
- With probability $\epsilon$ (epsilon): **Explore** — choose a random action
- With probability $1 - \epsilon$: **Exploit** — choose the action with highest Q-value

Typically, $\epsilon$ starts high (e.g., 1.0) and **decays** over time as the agent learns.


In [ ]:
# Visualizing the Exploration vs Exploitation trade-off
episodes = np.arange(0, 1000)
epsilon_decay = 1.0 * np.exp(-episodes / 200)  # Exponential decay

plt.figure(figsize=(10, 5))
plt.plot(episodes, epsilon_decay, linewidth=2.5, color='#FF6B6B', label='Epsilon (ε)')
plt.axhline(y=0.1, color='#4ECDC4', linestyle='--', linewidth=2, label='Minimum ε = 0.1')
plt.fill_between(episodes, epsilon_decay, alpha=0.3, color='#FF6B6B')
plt.title('Epsilon Decay Over Episodes: From Exploration to Exploitation', fontsize=14, fontweight='bold')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Epsilon Value', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

print("📉 As training progresses, the agent shifts from exploring randomly to exploiting known good strategies.")


## 4. Q-Learning Algorithm Explained 🧠

Q-Learning is a **model-free**, **off-policy** reinforcement learning algorithm that learns the value of actions in states. It was introduced by Chris Watkins in 1989 and remains one of the most important algorithms in RL.

### The Q-Table
The Q-Table is a lookup table where rows represent states and columns represent actions. Each cell $Q(s,a)$ stores the expected cumulative reward of taking action $a$ in state $s$.

### The Bellman Equation
The foundation of Q-Learning is the Bellman optimality equation:
$$Q^*(s, a) = R(s, a) + \gamma \max_{a'} Q^*(s', a')$$

Where:
- $Q^*(s, a)$ is the optimal Q-value
- $R(s, a)$ is the immediate reward
- $\gamma$ (gamma) is the discount factor (0 ≤ γ ≤ 1)
- $s'$ is the next state
- $\max_{a'} Q^*(s', a')$ is the best future value

### The Q-Learning Update Rule
Since we don't know $Q^*$ initially, we iteratively update our estimates:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Where:
- $\alpha$ (alpha) is the **learning rate** (how quickly we update our beliefs)
- $r$ is the reward received
- The term in brackets is the **Temporal Difference (TD) Error**

### Algorithm Steps:
1. Initialize Q-Table with zeros (or small random values)
2. For each episode:
   3. Observe current state $s$
   4. Choose action $a$ using epsilon-greedy policy
   5. Take action $a$, observe reward $r$ and new state $s'$
   6. Update $Q(s, a)$ using the update rule
   7. Set $s \leftarrow s'$ and repeat until terminal state


In [ ]:
# Let's implement the Q-Learning update rule as a function for clarity
def q_learning_update(Q, state, action, reward, next_state, alpha=0.1, gamma=0.95):
    """
    Perform a single Q-Learning update.
    
    Parameters:
    -----------
    Q : dict or np.array
        The Q-Table
    state : int or tuple
        Current state
    action : int
        Action taken
    reward : float
        Reward received
    next_state : int or tuple
        Next state after taking action
    alpha : float
        Learning rate (0 < alpha <= 1)
    gamma : float
        Discount factor (0 <= gamma <= 1)
    
    Returns:
    --------
    float : The updated Q-value
    """
    # Current Q-value
    current_q = Q[state][action]
    
    # Maximum Q-value for the next state (best future action)
    max_next_q = np.max(Q[next_state])
    
    # Calculate Temporal Difference (TD) Error
    td_error = reward + gamma * max_next_q - current_q
    
    # Update Q-value
    new_q = current_q + alpha * td_error
    
    Q[state][action] = new_q
    
    return new_q

print("✅ Q-Learning update function defined!")
print("📐 Formula: Q(s,a) = Q(s,a) + α * [r + γ * max(Q(s',a')) - Q(s,a)]")


## 5. Implementing Q-Learning from Scratch on a Simple Grid World 🗺️

Let's build a custom **Grid World** environment! Our agent starts at the top-left corner and must navigate to the goal at the bottom-right corner, avoiding pitfalls along the way.

### Environment Design:
- **Grid Size:** 5x5
- **Start:** Top-left (0, 0)
- **Goal:** Bottom-right (4, 4) — Reward: +10
- **Pitfalls:** Two trap cells — Reward: -10
- **Normal Move:** Reward: -1 (encourages finding the shortest path)
- **Actions:** Up (0), Down (1), Left (2), Right (3)


In [ ]:
class GridWorld:
    """
    A simple 5x5 Grid World environment for Q-Learning.
    
    The agent must navigate from start to goal while avoiding pitfalls.
    """
    def __init__(self, size=5):
        self.size = size
        self.start_state = (0, 0)
        self.goal_state = (4, 4)
        self.pitfalls = [(2, 2), (3, 3)]  # Danger zones!
        self.state = self.start_state
        self.actions = [0, 1, 2, 3]  # Up, Down, Left, Right
        self.action_names = ['↑ Up', '↓ Down', '← Left', '→ Right']
        
    def reset(self):
        """Reset the environment to the start state."""
        self.state = self.start_state
        return self.state
    
    def step(self, action):
","        """
        Execute an action and return (next_state, reward, done, info).
        
        Actions:
        0: Up, 1: Down, 2: Left, 3: Right
        """
        row, col = self.state
        
        # Calculate next position based on action
        if action == 0:    # Up
            row = max(0, row - 1)
        elif action == 1:  # Down
            row = min(self.size - 1, row + 1)
        elif action == 2:  # Left
            col = max(0, col - 1)
        elif action == 3:  # Right
            col = min(self.size - 1, col + 1)
        
        next_state = (row, col)
        self.state = next_state
        
        # Determine reward and if episode is done
        if next_state == self.goal_state:
            reward = 10.0
            done = True
        elif next_state in self.pitfalls:
            reward = -10.0
            done = True
        else:
            reward = -1.0  # Small penalty for each step to encourage efficiency
            done = False
        
        return next_state, reward, done, {}
    
    def render(self):
        """Visualize the current state of the grid."""
        grid = np.zeros((self.size, self.size), dtype=str)
        grid[:] = '·'
        
        for pit in self.pitfalls:
            grid[pit] = '💀'
        grid[self.goal_state] = '🏆'
        grid[self.state] = '🤖'
        
        print("\n" + "=" * (self.size * 4 + 1))
        for row in range(self.size):
            print("|", end="")
            for col in range(self.size):
                print(f" {grid[row, col]} |", end="")
            print("\n" + "=" * (self.size * 4 + 1))
        print()

# Test the environment
env = GridWorld()
print("🗺️ Grid World Environment Created!")
print("🤖 Agent starts at:", env.start_state)
print("🏆 Goal is at:", env.goal_state)
print("💀 Pitfalls at:", env.pitfalls)
env.render()


In [ ]:
# Initialize the Q-Table for our Grid World
# States are (row, col) tuples, actions are 0-3
def initialize_q_table(env):
    """Initialize Q-Table with zeros for all state-action pairs."""
    Q = {}
    for row in range(env.size):
        for col in range(env.size):
            Q[(row, col)] = np.zeros(len(env.actions))
    return Q

Q = initialize_q_table(env)
print("📊 Q-Table initialized!")
print("Shape:", env.size, "x", env.size, "states x", len(env.actions), "actions")
print("\nInitial Q-values for start state (0,0):", Q[(0, 0)])
print("Initial Q-values for goal state (4,4):", Q[(4, 4)])


In [ ]:
def epsilon_greedy_policy(Q, state, epsilon, n_actions):
    """
    Choose an action using the epsilon-greedy policy.
    
    With probability epsilon: choose random action (explore)
    With probability 1-epsilon: choose best action (exploit)
    """
    if random.random() < epsilon:
        # Explore: random action
        return random.randint(0, n_actions - 1)
    else:
        # Exploit: action with highest Q-value
        # If multiple actions have the same max value, choose randomly among them
        q_values = Q[state]
        max_q = np.max(q_values)
        best_actions = [a for a, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)

# Test the epsilon-greedy policy
test_state = (0, 0)
print("🎲 Testing Epsilon-Greedy Policy at state", test_state)
print("Q-values:", Q[test_state])

for eps in [1.0, 0.5, 0.0]:
    actions = [epsilon_greedy_policy(Q, test_state, eps, 4) for _ in range(10)]
    print(f"ε = {eps}: Actions chosen = {actions}")


## 6. Training Loop, Epsilon-Greedy Policy, and Convergence 🔄

Now comes the exciting part — let's train our agent! We'll run thousands of episodes where the agent explores the grid, learns from rewards, and gradually improves its strategy.

### Hyperparameters:
- **Learning Rate ($\alpha$):** 0.1 — How much we update Q-values each step
- **Discount Factor ($\gamma$):** 0.95 — How much we value future rewards
- **Epsilon ($\epsilon$):** Starts at 1.0, decays to 0.01 — Exploration rate
- **Episodes:** 5,000 — Total training episodes


In [ ]:
# Training hyperparameters
ALPHA = 0.1       # Learning rate
GAMMA = 0.95      # Discount factor
EPSILON = 1.0     # Initial exploration rate
EPSILON_DECAY = 0.995  # Decay rate per episode
EPSILON_MIN = 0.01     # Minimum exploration rate
N_EPISODES = 5000  # Total training episodes
MAX_STEPS = 100    # Maximum steps per episode

# Initialize fresh Q-Table and environment
env = GridWorld()
Q = initialize_q_table(env)

# Track metrics
rewards_history = []
epsilon_history = []
steps_history = []
q_table_snapshots = {}  # Store Q-table at different milestones

print("🚀 Starting Q-Learning Training...")
print("=" * 50)
print(f"Episodes: {N_EPISODES}")
print(f"Learning Rate (α): {ALPHA}")
print(f"Discount Factor (γ): {GAMMA}")
print(f"Initial Epsilon (ε): {EPSILON}")
print("=" * 50)


In [ ]:
# Main Training Loop
for episode in range(N_EPISODES):
    state = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < MAX_STEPS:
        # Choose action using epsilon-greedy policy
        action = epsilon_greedy_policy(Q, state, EPSILON, len(env.actions))
        
        # Take action in the environment
        next_state, reward, done, _ = env.step(action)
        
        # Q-Learning update
        old_q = Q[state][action]
        td_target = reward + GAMMA * np.max(Q[next_state]) * (0 if done else 1)
        td_error = td_target - old_q
        Q[state][action] = old_q + ALPHA * td_error
        
        state = next_state
        total_reward += reward
        steps += 1
    
    # Record metrics
    rewards_history.append(total_reward)
    epsilon_history.append(EPSILON)
    steps_history.append(steps)
    
    # Decay epsilon
    EPSILON = max(EPSILON_MIN, EPSILON * EPSILON_DECAY)
    
    # Save Q-table snapshots at milestones
    if episode in [0, 99, 499, 999, 2499, 4999]:
        q_table_snapshots[episode + 1] = {k: v.copy() for k, v in Q.items()}
    
    # Progress update every 1000 episodes
    if (episode + 1) % 1000 == 0:
        avg_reward = np.mean(rewards_history[-1000:])
        print(f"Episode {episode + 1:5d} | Avg Reward (last 1000): {avg_reward:7.2f} | ε: {EPSILON:.4f}")

print("\n✅ Training Complete!")


In [ ]:
# Plot training progress
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Rewards over episodes (smoothed)
window = 100
smoothed_rewards = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
axes[0].plot(range(window, N_EPISODES + 1), smoothed_rewards, color='#FF6B6B', linewidth=2)
axes[0].set_title('Total Reward per Episode (Smoothed)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# 2. Epsilon decay
axes[1].plot(epsilon_history, color='#4ECDC4', linewidth=2)
axes[1].set_title('Epsilon Decay Over Time', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Epsilon (ε)')
axes[1].grid(True, alpha=0.3)

# 3. Steps per episode (smoothed)
smoothed_steps = np.convolve(steps_history, np.ones(window)/window, mode='valid')
axes[2].plot(range(window, N_EPISODES + 1), smoothed_steps, color='#95E1D3', linewidth=2)
axes[2].set_title('Steps per Episode (Smoothed)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Steps')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Notice how rewards improve and steps decrease as the agent learns!")


## 7. Visualizing the Learned Policy and Q-Table 🔥

Let's see what our agent has learned! We'll visualize:
1. The **Q-Table heatmap** showing value estimates for each state-action pair
2. The **learned policy** — the best action to take in each state
3. A comparison of Q-values at different training milestones


In [ ]:
# Visualize the final Q-Table as a heatmap
def plot_q_table_heatmap(Q, env, title="Q-Table Heatmap"):
    """Visualize Q-values for each action across the grid."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    action_names = ['Up', 'Down', 'Left', 'Right']
    
    for idx, (ax, action_name) in enumerate(zip(axes.flat, action_names)):
        q_grid = np.zeros((env.size, env.size))
        for row in range(env.size):
            for col in range(env.size):
                q_grid[row, col] = Q[(row, col)][idx]
        
        sns.heatmap(q_grid, annot=True, fmt='.2f', cmap='RdYlGn', 
                   center=0, ax=ax, cbar_kws={'label': 'Q-Value'},
                   linewidths=0.5, linecolor='gray')
        ax.set_title(f'Action: {action_name}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_q_table_heatmap(Q, env, "Final Learned Q-Table (After 5000 Episodes)")


In [ ]:
# Visualize the learned policy (best action in each state)
def plot_policy(Q, env):
    """Display the optimal policy as arrows in the grid."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Create grid
    for row in range(env.size + 1):
        ax.axhline(row, color='gray', linewidth=0.5)
    for col in range(env.size + 1):
        ax.axvline(col, color='gray', linewidth=0.5)
    
    # Arrow directions
    arrows = {0: (0, -0.3), 1: (0, 0.3), 2: (-0.3, 0), 3: (0.3, 0)}  # Up, Down, Left, Right
    
    for row in range(env.size):
        for col in range(env.size):
            state = (row, col)
            
            # Mark special cells
            if state == env.goal_state:
                ax.text(col + 0.5, row + 0.5, '🏆', fontsize=25, ha='center', va='center')
                continue
            elif state in env.pitfalls:
                ax.text(col + 0.5, row + 0.5, '💀', fontsize=25, ha='center', va='center')
                continue
            elif state == env.start_state:
                ax.text(col + 0.5, row + 0.5, '🚀', fontsize=20, ha='center', va='center')
            
            # Get best action
            best_action = np.argmax(Q[state])
            dx, dy = arrows[best_action]
            
            # Draw arrow
            ax.arrow(col + 0.5 - dx/2, row + 0.5 - dy/2, dx, dy, 
                    head_width=0.15, head_length=0.1, fc='#FF6B6B', ec='#FF6B6B', linewidth=2)
    
    ax.set_xlim(0, env.size)
    ax.set_ylim(env.size, 0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Learned Optimal Policy\n(Arrows indicate best action)', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

plot_policy(Q, env)
print("🧭 The arrows show the optimal direction the agent should take from each cell!")


In [ ]:
# Compare Q-Table evolution at different milestones
milestones = [1, 100, 500, 1000, 2500, 5000]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, milestone in enumerate(milestones):
    if milestone not in q_table_snapshots:
        continue
    
    Q_snap = q_table_snapshots[milestone]
    # Calculate max Q-value for each state
    max_q_grid = np.zeros((env.size, env.size))
    for row in range(env.size):
        for col in range(env.size):
            max_q_grid[row, col] = np.max(Q_snap[(row, col)])
    
    sns.heatmap(max_q_grid, annot=True, fmt='.1f', cmap='viridis', 
               ax=axes[idx], cbar_kws={'label': 'Max Q-Value'},
               linewidths=0.5, linecolor='gray')
    axes[idx].set_title(f'After {milestone} Episodes', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Column')
    axes[idx].set_ylabel('Row')

plt.suptitle('Q-Table Evolution: Maximum Q-Value per State', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 Watch how the agent's understanding of state values improves over time!")


## 8. Applying Q-Learning to a Classic Environment 🎮

Now that we've mastered Q-Learning on our custom grid, let's apply it to a classic RL environment from **OpenAI Gym/Gymnasium**. We'll use **FrozenLake-v1**, where the agent must navigate across a frozen lake to reach a goal without falling through holes.

### FrozenLake Environment:
- **S:** Start (safe)
- **F:** Frozen surface (safe)
- **H:** Hole (game over, reward 0)
- **G:** Goal (reward +1)
- **Actions:** Left (0), Down (1), Right (2), Up (3)
- **Slippery:** The ice is slippery! Sometimes you move perpendicular to your intended direction.


In [ ]:
# Install gymnasium if needed and import
try:
    import gymnasium as gym
    print("✅ Using Gymnasium (recommended)")
except ImportError:
    try:
        import gym
        print("✅ Using OpenAI Gym")
    except ImportError:
        print("📦 Installing gymnasium...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium"])
        import gymnasium as gym
        print("✅ Gymnasium installed and imported!")

# Create the FrozenLake environment
env_frozen = gym.make('FrozenLake-v1', desc=None, map_name="4x4", is_slippery=True, render_mode=None)
print("\n🧊 FrozenLake Environment Created!")
print("Observation Space:", env_frozen.observation_space)
print("Action Space:", env_frozen.action_space)
print("\nMap Layout:")
print("S = Start, F = Frozen, H = Hole, G = Goal")


In [ ]:
# Visualize the FrozenLake map
def print_frozen_lake_map():
    """Print the FrozenLake 4x4 map."""
    map_desc = [
        "SFFF",
        "FHFH", 
        "FFFH",
        "HFFG"
    ]
    print("\n🗺️ FrozenLake 4x4 Map:")
    print("=" * 17)
    for i, row in enumerate(map_desc):
        print("|", end="")
        for j, cell in enumerate(row):
            if cell == 'S':
                symbol = '🚀'
            elif cell == 'F':
                symbol = '❄️'
            elif cell == 'H':
                symbol = '🕳️'
            elif cell == 'G':
                symbol = '🏆'
            print(f" {symbol} |", end="")
        print()
        if i < len(map_desc) - 1:
            print("-" * 17)
    print("=" * 17)
    print("\nLegend: 🚀 Start | ❄️ Frozen | 🕳️ Hole | 🏆 Goal")

print_frozen_lake_map()


In [ ]:
# Train Q-Learning on FrozenLake
def train_q_learning_frozenlake(env, n_episodes=10000, alpha=0.1, gamma=0.99, 
                                epsilon=1.0, epsilon_decay=0.9995, epsilon_min=0.01):
    """Train Q-Learning agent on FrozenLake environment."""
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    # Initialize Q-Table
    Q = np.zeros((n_states, n_actions))
    
    rewards_history = []
    epsilon_history = []
    success_count = 0
    
    print(f"🚀 Training on FrozenLake for {n_episodes} episodes...")
    print("=" * 50)
    
    for episode in range(n_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                # Break ties randomly
                max_q = np.max(Q[state])
                best_actions = [a for a in range(n_actions) if Q[state][a] == max_q]
                action = random.choice(best_actions)
            
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Q-Learning update
            old_q = Q[state][action]
            td_target = reward + gamma * np.max(Q[next_state]) * (0 if done else 1)
            Q[state][action] = old_q + alpha * (td_target - old_q)
            
            state = next_state
            total_reward += reward
        
        if total_reward > 0:
            success_count += 1
            
        rewards_history.append(total_reward)
        epsilon_history.append(epsilon)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        
        if (episode + 1) % 2000 == 0:
            success_rate = success_count / (episode + 1) * 100
            print(f"Episode {episode + 1:5d} | Success Rate: {success_rate:5.1f}% | ε: {epsilon:.4f}")
    
    print("=" * 50)
    print(f"✅ Training Complete! Final Success Rate: {success_count/n_episodes*100:.1f}%")
    
    return Q, rewards_history, epsilon_history

Q_frozen, rewards_frozen, epsilon_frozen = train_q_learning_frozenlake(env_frozen)


In [ ]:
# Plot FrozenLake training results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Smoothed success rate
window = 500
smoothed_rewards = np.convolve(rewards_frozen, np.ones(window)/window, mode='valid')
axes[0].plot(range(window, len(rewards_frozen) + 1), smoothed_rewards * 100, 
            color='#4ECDC4', linewidth=2)
axes[0].set_title('FrozenLake: Success Rate Over Episodes (Smoothed)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Success Rate (%)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 105)

# Epsilon decay
axes[1].plot(epsilon_frozen, color='#FF6B6B', linewidth=2)
axes[1].set_title('Epsilon Decay', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Epsilon')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize the learned policy for FrozenLake
def visualize_frozenlake_policy(Q, env):
    """Visualize the learned policy for FrozenLake."""
    map_desc = [
        "SFFF",
        "FHFH",
        "FFFH", 
        "HFFG"
    ]
    
    action_arrows = ['←', '↓', '→', '↑']  # Gymnasium actions: Left, Down, Right, Up
    
    print("\n🧭 Learned Policy for FrozenLake:")
    print("=" * 25)
    
    for row in range(4):
        policy_row = "|"
        for col in range(4):
            state = row * 4 + col
            cell = map_desc[row][col]
            
            if cell in ['H', 'G']:
                symbol = '🕳️' if cell == 'H' else '🏆'
                policy_row += f" {symbol} |"
            else:
                best_action = np.argmax(Q[state])
                policy_row += f"  {action_arrows[best_action]}  |"
        print(policy_row)
        if row < 3:
            print("-" * 25)
    print("=" * 25)
    
    # Print Q-Table values
    print("\n📊 Q-Table Values:")
    print("State | Left    | Down    | Right   | Up")
    print("-" * 50)
    for state in range(16):
        print(f"  {state:2d}  | {Q[state][0]:6.3f} | {Q[state][1]:6.3f} | {Q[state][2]:6.3f} | {Q[state][3]:6.3f}")

visualize_frozenlake_policy(Q_frozen, env_frozen)


In [ ]:
# Compare Q-Learning agent vs Random agent on FrozenLake
def evaluate_agent(env, Q=None, n_episodes=1000):
    """Evaluate an agent (Q-Learning or Random) over multiple episodes."""
    successes = 0
    total_rewards = []
    
    for _ in range(n_episodes):
        state, info = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            if Q is not None:
                # Q-Learning agent
                action = np.argmax(Q[state])
            else:
                # Random agent
                action = env.action_space.sample()
            
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        if episode_reward > 0:
            successes += 1
        total_rewards.append(episode_reward)
    
    return successes / n_episodes * 100, np.mean(total_rewards)

# Evaluate both agents
q_success, q_avg_reward = evaluate_agent(env_frozen, Q_frozen, 1000)
random_success, random_avg_reward = evaluate_agent(env_frozen, None, 1000)

print("📊 Performance Comparison (1000 episodes each):")
print("=" * 45)
print(f"🤖 Q-Learning Agent:  {q_success:5.1f}% success | Avg Reward: {q_avg_reward:.3f}")
print(f"🎲 Random Agent:      {random_success:5.1f}% success | Avg Reward: {random_avg_reward:.3f}")
print("=" * 45)
print(f"💡 Q-Learning is {q_success/max(random_success, 0.1):.1f}x better than random!")


## 9. Limitations of Tabular Q-Learning and the Road to Deep RL 🛣️

While Q-Learning is powerful and intuitive, it has significant limitations that motivated the development of Deep Reinforcement Learning:

### 🔴 Limitations of Tabular Q-Learning:
1. **Curse of Dimensionality:** The Q-Table size grows exponentially with state space. A game with just 10 binary features has $2^{10} = 1,024$ states. Real-world problems have millions or billions of states.
2. **Discrete States Only:** Q-Tables require discrete, enumerable states. Continuous state spaces (like robot joint angles or stock prices) cannot be directly represented.
3. **No Generalization:** The agent learns about each state independently. It doesn't transfer knowledge from similar states.
4. **Memory Intensive:** Storing a massive Q-Table becomes impractical.
5. **Slow Convergence:** In large environments, it can take prohibitively long to visit every state-action pair enough times.

### 🟢 The Solution: Deep Q-Networks (DQN)
Instead of a lookup table, we use a **neural network** to approximate the Q-function:
$$Q(s, a; \theta) \approx Q^*(s, a)$$

**Key Innovations:**
- **Experience Replay:** Store and sample past experiences to break correlation
- **Target Networks:** Use a separate network to stabilize learning
- **Function Approximation:** Generalize across similar states

This is exactly what we'll explore in **Day 73: Deep Q-Networks (DQN)**! 🚀


In [ ]:
# Visual comparison: Tabular vs Deep RL
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Tabular Q-Learning visualization
ax1 = axes[0]
ax1.text(0.5, 0.8, 'Tabular Q-Learning', ha='center', fontsize=14, fontweight='bold', transform=ax1.transAxes)
ax1.text(0.5, 0.65, 'Q-Table: States × Actions', ha='center', fontsize=11, transform=ax1.transAxes)
ax1.text(0.5, 0.55, '❌ Memory: O(|S| × |A|)', ha='center', fontsize=10, transform=ax1.transAxes)
ax1.text(0.5, 0.45, '❌ No Generalization', ha='center', fontsize=10, transform=ax1.transAxes)
ax1.text(0.5, 0.35, '❌ Discrete States Only', ha='center', fontsize=10, transform=ax1.transAxes)
ax1.text(0.5, 0.25, '✅ Simple & Interpretable', ha='center', fontsize=10, color='green', transform=ax1.transAxes)
ax1.text(0.5, 0.15, '✅ Guaranteed Convergence', ha='center', fontsize=10, color='green', transform=ax1.transAxes)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Traditional Approach', fontsize=12, pad=10)

# Deep Q-Network visualization
ax2 = axes[1]
ax2.text(0.5, 0.8, 'Deep Q-Network (DQN)', ha='center', fontsize=14, fontweight='bold', transform=ax2.transAxes)
ax2.text(0.5, 0.65, 'Neural Network: Q(s,a; θ)', ha='center', fontsize=11, transform=ax2.transAxes)
ax2.text(0.5, 0.55, '✅ Memory: O(Parameters)', ha='center', fontsize=10, color='green', transform=ax2.transAxes)
ax2.text(0.5, 0.45, '✅ Generalizes Across States', ha='center', fontsize=10, color='green', transform=ax2.transAxes)
ax2.text(0.5, 0.35, '✅ Continuous States OK', ha='center', fontsize=10, color='green', transform=ax2.transAxes)
ax2.text(0.5, 0.25, '❌ Requires More Data', ha='center', fontsize=10, transform=ax2.transAxes)
ax2.text(0.5, 0.15, '❌ Harder to Debug', ha='center', fontsize=10, transform=ax2.transAxes)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Modern Approach', fontsize=12, pad=10)

plt.suptitle('Tabular Q-Learning vs Deep Q-Networks', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("🚀 Tomorrow we bridge this gap with Deep Q-Networks!")


## 10. Pros, Cons, and Real-World Applications of RL 🌍

### ✅ Advantages of Reinforcement Learning:
- **Learns Optimal Strategies:** Can discover solutions humans might not think of
- **Handles Complex Environments:** Works in scenarios where rules are hard to define
- **Adaptive:** Can adjust to changing environments
- **Goal-Oriented:** Naturally optimizes for long-term objectives
- **Minimal Supervision:** Doesn't require labeled datasets

### ❌ Challenges of Reinforcement Learning:
- **Sample Inefficiency:** Requires massive amounts of interaction data
- **Reward Design:** Crafting good reward functions is difficult (reward hacking)
- **Unstable Training:** Can be sensitive to hyperparameters
- **Safety Concerns:** Agents might learn dangerous behaviors during exploration
- **Reproducibility:** Randomness in environments makes results hard to reproduce

### 🌟 Real-World Applications:
| Domain | Application |
|--------|-------------|
| **🎮 Gaming** | AlphaGo, AlphaStar, OpenAI Five (Dota 2) |
| **🤖 Robotics** | Robot locomotion, manipulation, autonomous navigation |
| **🚗 Autonomous Vehicles** | Self-driving car decision-making |
| **💰 Finance** | Algorithmic trading, portfolio optimization |
| **🏥 Healthcare** | Treatment optimization, drug discovery |
| **⚡ Energy** | Smart grid optimization, data center cooling |
| **📦 Logistics** | Supply chain optimization, warehouse robotics |
| **✈️ Aerospace** | Flight control systems, drone navigation |


In [ ]:
# Create a summary visualization of RL applications
applications = {
    '🎮 Gaming': 95,
    '🤖 Robotics': 85,
    '🚗 Autonomous Vehicles': 80,
    '💰 Finance': 75,
    '📦 Logistics': 70,
    '⚡ Energy': 65,
    '🏥 Healthcare': 60,
    '✈️ Aerospace': 55
}

fig, ax = plt.subplots(figsize=(10, 6))
categories = list(applications.keys())
values = list(applications.values())
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(categories)))

bars = ax.barh(categories, values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlim(0, 100)
ax.set_xlabel('Maturity / Impact Score', fontsize=12)
ax.set_title('Reinforcement Learning: Real-World Applications', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# Add value labels
for bar, val in zip(bars, values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val}', 
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("🌍 RL is transforming industries across the board!")


---
## 🛠️ Hands-On Exercises

Now it's your turn! These exercises will solidify your understanding of Q-Learning and Reinforcement Learning fundamentals. Try each one before checking the solutions.


### Exercise 1: Implement Epsilon-Greedy from Scratch
Write a function `epsilon_greedy(q_values, epsilon)` that takes a list of Q-values for a state and an epsilon value, and returns an action index using the epsilon-greedy strategy. Test it with different epsilon values and observe the behavior.


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Modify the Grid World Environment
Create a new Grid World with different dimensions (e.g., 6x6), add more pitfalls, and place the goal in a different location. Train your Q-Learning agent on this new environment and compare the learning curve with the original 5x5 grid.


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Experiment with Learning Rate and Discount Factor
Train multiple Q-Learning agents on the same environment with different combinations of `alpha` (0.01, 0.1, 0.5, 1.0) and `gamma` (0.5, 0.9, 0.99). Plot the learning curves together and analyze which combination works best. What happens when gamma is 0? What happens when alpha is too high?


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Visualize Q-Table Evolution During Training
Modify the training loop to save the Q-Table every 500 episodes. Create an animation or a series of heatmaps showing how the Q-values for a specific state (e.g., the start state) change over time. What patterns do you observe?


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Solve FrozenLake with Q-Learning
Use the Q-Learning implementation from this notebook to solve the `FrozenLake8x8-v1` environment (the larger 8x8 version). How many episodes does it take to converge? Is it harder than the 4x4 version? Why?


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Compare Q-Learning with a Random Policy
Implement a systematic comparison between your trained Q-Learning agent and a purely random agent across 1,000 test episodes on FrozenLake. Calculate and compare: average reward, success rate, average episode length, and reward variance. Create a bar chart comparing these metrics.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Implement a Decaying Learning Rate
Currently, our implementation uses a fixed learning rate. Modify the training loop to decay the learning rate over time (e.g., `alpha = alpha_min + (alpha_max - alpha_min) * exp(-episode / decay_rate)`). Does this improve convergence? Test and compare with the fixed learning rate approach.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Create a Custom Reward Function
Design a more sophisticated reward function for the Grid World environment. For example:
- Give a small bonus for moving closer to the goal (Manhattan distance heuristic)
- Penalize revisiting the same state multiple times
- Give a time bonus for reaching the goal quickly
Train the agent with your custom rewards and compare the learned policy with the original.


In [ ]:
# Exercise 8: Your code here



### Exercise 9: Extend to Larger State Spaces
Discuss and implement a simple discretization strategy for a continuous state space. For example, create a MountainCar-like environment where the state has continuous position and velocity. Discretize these continuous values into bins and apply Q-Learning. What challenges do you face as the number of bins increases?


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Implement SARSA Algorithm
SARSA (State-Action-Reward-State-Action) is another fundamental RL algorithm. Unlike Q-Learning (off-policy), SARSA is on-policy — it uses the actual next action taken to update Q-values. The update rule is:
$$Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma Q(s', a') - Q(s, a)]$$
Implement SARSA and compare its performance with Q-Learning on the Grid World environment. Which converges faster? Which achieves better final performance?


In [ ]:
# Exercise 10: Your code here



---
## Solutions (Review After Attempting) 🔑

Below are detailed solutions and explanations for each exercise. **We strongly encourage you to attempt each exercise on your own before reviewing these solutions.** The learning happens in the struggle! 💪


### Solution 1: Epsilon-Greedy Implementation

**Concept:** The epsilon-greedy strategy balances exploration and exploitation. With probability `epsilon`, we choose a random action (explore). Otherwise, we choose the action with the highest estimated Q-value (exploit).

**Key Points:**
- When `epsilon = 1.0`, the policy is completely random
- When `epsilon = 0.0`, the policy is completely greedy
- Breaking ties randomly among equally-valued actions prevents bias


In [ ]:
# Solution 1: Epsilon-Greedy Implementation
import random
import numpy as np

def epsilon_greedy(q_values, epsilon):
    """
    Select an action using epsilon-greedy policy.
    
    Parameters:
    -----------
    q_values : list or np.array
        Q-values for each action in the current state
    epsilon : float
        Probability of taking a random action (0 <= epsilon <= 1)
    
    Returns:
    --------
    int : Index of the chosen action
    """
    n_actions = len(q_values)
    
    if random.random() < epsilon:
        # Explore: choose random action
        return random.randint(0, n_actions - 1)
    else:
        # Exploit: choose action with highest Q-value
        max_q = np.max(q_values)
        # Break ties randomly
        best_actions = [i for i, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)

# Test the function
test_q = [0.5, 1.2, 0.8, 0.3]
print("Q-values:", test_q)
print("Best action (ε=0.0):", epsilon_greedy(test_q, 0.0))
print("Random action (ε=1.0):", epsilon_greedy(test_q, 1.0))

# Test with multiple runs at ε=0.5
results = [epsilon_greedy(test_q, 0.5) for _ in range(20)]
print("\n20 runs with ε=0.5:", results)
print("Exploitation rate:", sum(1 for r in results if r == 1) / 20 * 100, "%")


### Solution 2: Modified Grid World Environment

**Concept:** Creating custom environments helps you understand how RL agents adapt to different problem structures. A larger grid with more obstacles tests the agent's ability to plan longer trajectories.

**Key Modifications:**
- Increased grid size to 6x6
- Added more pitfalls for complexity
- Changed goal position to test generalization
- Maintained consistent reward structure


In [ ]:
# Solution 2: Modified Grid World (6x6)
class GridWorld6x6:
    def __init__(self):
        self.size = 6
        self.start_state = (0, 0)
        self.goal_state = (5, 5)
        self.pitfalls = [(1, 3), (2, 2), (3, 4), (4, 1), (4, 3)]
        self.state = self.start_state
        self.actions = [0, 1, 2, 3]  # Up, Down, Left, Right
        
    def reset(self):
        self.state = self.start_state
        return self.state
    
    def step(self, action):
        row, col = self.state
        
        if action == 0: row = max(0, row - 1)
        elif action == 1: row = min(self.size - 1, row + 1)
        elif action == 2: col = max(0, col - 1)
        elif action == 3: col = min(self.size - 1, col + 1)
        
        next_state = (row, col)
        self.state = next_state
        
        if next_state == self.goal_state:
            reward, done = 10.0, True
        elif next_state in self.pitfalls:
            reward, done = -10.0, True
        else:
            reward, done = -1.0, False
        
        return next_state, reward, done, {}

# Train on 6x6 grid
env6 = GridWorld6x6()
Q6 = {(r, c): np.zeros(4) for r in range(6) for c in range(6)}

epsilon = 1.0
rewards_6x6 = []
for episode in range(8000):
    state = env6.reset()
    total_reward = 0
    done = False
    steps = 0
    
    while not done and steps < 150:
        action = epsilon_greedy(Q6[state], epsilon, 4)
        next_state, reward, done, _ = env6.step(action)
        
        # Q-Learning update
        Q6[state][action] += 0.1 * (reward + 0.95 * np.max(Q6[next_state]) * (0 if done else 1) - Q6[state][action])
        
        state = next_state
        total_reward += reward
        steps += 1
    
    rewards_6x6.append(total_reward)
    epsilon = max(0.01, epsilon * 0.995)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
window = 200
ax.plot(np.convolve(rewards_history, np.ones(window)/window, mode='valid'), 
        label='5x5 Grid', linewidth=2)
ax.plot(np.convolve(rewards_6x6, np.ones(window)/window, mode='valid'), 
        label='6x6 Grid', linewidth=2)
ax.set_title('Learning Curve Comparison: 5x5 vs 6x6 Grid World', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 The 6x6 grid takes longer to converge due to the larger state space!")


### Solution 3: Hyperparameter Tuning

**Concept:** The learning rate (`alpha`) controls how much we update our Q-values with new information. The discount factor (`gamma`) determines how much we care about future rewards versus immediate rewards.

**Insights:**
- **High alpha (1.0):** Agent is too reactive, can oscillate and never converge
- **Low alpha (0.01):** Learning is very slow but stable
- **Gamma = 0:** Agent is completely myopic, only cares about immediate reward
- **High gamma (0.99):** Agent plans far into the future, better for sparse rewards


In [ ]:
# Solution 3: Hyperparameter Comparison
def train_with_params(env, alpha, gamma, n_episodes=3000):
    """Train Q-Learning with specific hyperparameters."""
    Q = {(r, c): np.zeros(4) for r in range(env.size) for c in range(env.size)}
    epsilon = 1.0
    rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            action = epsilon_greedy(Q[state], epsilon, 4)
            next_state, reward, done, _ = env.step(action)
            Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) * (0 if done else 1) - Q[state][action])
            state = next_state
            total_reward += reward
            steps += 1
        
        rewards.append(total_reward)
        epsilon = max(0.01, epsilon * 0.995)
    
    return rewards

# Test different combinations
env_test = GridWorld()
alphas = [0.01, 0.1, 0.5, 1.0]
gammas = [0.5, 0.9, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, alpha in enumerate(alphas):
    for gamma in gammas:
        rewards = train_with_params(env_test, alpha, gamma, 3000)
        window = 100
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[idx].plot(smoothed, label=f'γ={gamma}', linewidth=2)
    
    axes[idx].set_title(f'α = {alpha}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Episode')
    axes[idx].set_ylabel('Smoothed Reward')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Q-Learning Performance Across Hyperparameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 α=0.1 and γ=0.9 typically provide a good balance for this environment!")


### Solution 4: Q-Table Evolution Visualization

**Concept:** Watching how Q-values evolve provides deep insight into the learning process. Early in training, Q-values are noisy and exploratory. As training progresses, they stabilize to reflect the true value of each action.

**Observations to Look For:**
- Initially, all Q-values are near zero
- States close to the goal develop high values first
- Value propagation spreads backward from the goal over time
- Eventually, the optimal path emerges with clearly distinct Q-values


In [ ]:
# Solution 4: Q-Table Evolution
env_evo = GridWorld()
Q_evo = {(r, c): np.zeros(4) for r in range(5) for c in range(5)}
epsilon = 1.0
snapshots = {}
start_state_q_history = []

for episode in range(5000):
    state = env_evo.reset()
    done = False
    steps = 0
    
    while not done and steps < 100:
        action = epsilon_greedy(Q_evo[state], epsilon, 4)
        next_state, reward, done, _ = env_evo.step(action)
        Q_evo[state][action] += 0.1 * (reward + 0.95 * np.max(Q_evo[next_state]) * (0 if done else 1) - Q_evo[state][action])
        state = next_state
        steps += 1
    
    # Record start state Q-values
    start_state_q_history.append(Q_evo[(0, 0)].copy())
    
    if (episode + 1) % 500 == 0:
        snapshots[episode + 1] = {k: v.copy() for k, v in Q_evo.items()}
    
    epsilon = max(0.01, epsilon * 0.995)

# Plot evolution of start state Q-values
start_q_history = np.array(start_state_q_history)
fig, ax = plt.subplots(figsize=(12, 6))
actions = ['Up', 'Down', 'Left', 'Right']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for i in range(4):
    ax.plot(start_q_history[:, i], label=f'{actions[i]}', color=colors[i], linewidth=2, alpha=0.8)

ax.set_title('Q-Value Evolution for Start State (0,0)', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Q-Value')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("📈 Notice how 'Down' and 'Right' (toward the goal) develop higher values!")


### Solution 5: FrozenLake 8x8

**Concept:** The 8x8 version of FrozenLake has a much larger state space (64 states vs 16) and is significantly harder due to the increased distance to the goal and more holes. The slippery ice adds stochasticity, making it harder to learn a reliable policy.

**Key Insight:** Tabular Q-Learning can still solve this, but it requires more episodes and careful hyperparameter tuning. The success rate is typically lower than the 4x4 version due to increased difficulty.


In [ ]:
# Solution 5: FrozenLake 8x8
env_8x8 = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True, render_mode=None)
Q_8x8, rewards_8x8, epsilon_8x8 = train_q_learning_frozenlake(
    env_8x8, n_episodes=20000, alpha=0.2, gamma=0.99, 
    epsilon=1.0, epsilon_decay=0.9995
)

# Compare with 4x4
fig, ax = plt.subplots(figsize=(10, 5))
window = 500
ax.plot(np.convolve(rewards_frozen, np.ones(window)/window, mode='valid') * 100, 
        label='FrozenLake 4x4', linewidth=2)
ax.plot(np.convolve(rewards_8x8, np.ones(window)/window, mode='valid') * 100, 
        label='FrozenLake 8x8', linewidth=2)
ax.set_title('FrozenLake: 4x4 vs 8x8 Comparison', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("🧊 The 8x8 version is significantly harder due to the larger state space!")


### Solution 6: Systematic Comparison with Random Policy

**Concept:** Comparing with a random baseline is essential in RL to verify that learning is actually occurring. A well-trained agent should significantly outperform random action selection.

**Metrics to Compare:**
- Success rate (most important)
- Average reward
- Episode length (shorter = more efficient)
- Consistency (variance)


In [ ]:
# Solution 6: Systematic Comparison
def comprehensive_evaluation(env, Q=None, n_episodes=1000):
    """Comprehensive evaluation of an agent."""
    successes = 0
    rewards = []
    lengths = []
    
    for _ in range(n_episodes):
        state, info = env.reset()
        done = False
        episode_reward = 0
        steps = 0
        
        while not done:
            if Q is not None:
                action = np.argmax(Q[state])
            else:
                action = env.action_space.sample()
            
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            steps += 1
        
        if episode_reward > 0:
            successes += 1
        rewards.append(episode_reward)
        lengths.append(steps)
    
    return {
        'success_rate': successes / n_episodes * 100,
        'avg_reward': np.mean(rewards),
        'avg_length': np.mean(lengths),
        'reward_std': np.std(rewards)
    }

# Evaluate both
q_metrics = comprehensive_evaluation(env_frozen, Q_frozen, 1000)
random_metrics = comprehensive_evaluation(env_frozen, None, 1000)

# Create comparison chart
metrics = ['Success Rate (%)', 'Avg Reward', 'Avg Length', 'Reward Std']
q_values = [q_metrics['success_rate'], q_metrics['avg_reward'], 
            q_metrics['avg_length'], q_metrics['reward_std']]
random_values = [random_metrics['success_rate'], random_metrics['avg_reward'],
                 random_metrics['avg_length'], random_metrics['reward_std']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, q_values, width, label='Q-Learning', color='#4ECDC4', edgecolor='black')
bars2 = ax.bar(x + width/2, random_values, width, label='Random', color='#FF6B6B', edgecolor='black')

ax.set_ylabel('Value')
ax.set_title('Q-Learning vs Random Agent: Comprehensive Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Q-Learning significantly outperforms the random baseline across all metrics!")


### Solution 7: Decaying Learning Rate

**Concept:** A decaying learning rate allows the agent to make large updates early in training (when Q-values are very wrong) and smaller updates later (when Q-values are close to optimal). This can improve stability and convergence.

**Formula:** `alpha = alpha_min + (alpha_max - alpha_min) * exp(-episode / decay_rate)`


In [ ]:
# Solution 7: Decaying Learning Rate
def train_with_decaying_alpha(env, n_episodes=5000, alpha_max=0.5, alpha_min=0.01, 
                               decay_rate=1000, gamma=0.95):
    """Train with decaying learning rate."""
    Q = {(r, c): np.zeros(4) for r in range(env.size) for c in range(env.size)}
    epsilon = 1.0
    rewards = []
    alphas = []
    
    for episode in range(n_episodes):
        # Decay alpha
        alpha = alpha_min + (alpha_max - alpha_min) * np.exp(-episode / decay_rate)
        alphas.append(alpha)
        
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            action = epsilon_greedy(Q[state], epsilon, 4)
            next_state, reward, done, _ = env.step(action)
            Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) * (0 if done else 1) - Q[state][action])
            state = next_state
            total_reward += reward
            steps += 1
        
        rewards.append(total_reward)
        epsilon = max(0.01, epsilon * 0.995)
    
    return rewards, alphas

# Compare fixed vs decaying alpha
env_alpha = GridWorld()
rewards_fixed = train_with_params(env_alpha, 0.1, 0.95, 5000)
rewards_decay, alphas = train_with_decaying_alpha(env_alpha, 5000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 100
axes[0].plot(np.convolve(rewards_fixed, np.ones(window)/window, mode='valid'), 
             label='Fixed α=0.1', linewidth=2)
axes[0].plot(np.convolve(rewards_decay, np.ones(window)/window, mode='valid'), 
             label='Decaying α', linewidth=2)
axes[0].set_title('Learning Curve: Fixed vs Decaying Learning Rate', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Smoothed Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(alphas, color='#FF6B6B', linewidth=2)
axes[1].set_title('Learning Rate Decay Schedule', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Alpha (α)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📉 Decaying alpha often leads to more stable final performance!")


### Solution 8: Custom Reward Function

**Concept:** Reward shaping is a powerful technique to guide learning. By providing intermediate rewards that align with the ultimate goal, we can speed up learning and help the agent discover good policies faster.

**Important:** Reward shaping must be careful — poorly designed rewards can lead to unexpected behaviors (reward hacking).


In [ ]:
# Solution 8: Custom Reward Function
class GridWorldShaped(GridWorld):
    def step(self, action):
        """Modified step with shaped rewards."""
        row, col = self.state
        old_state = (row, col)
        
        if action == 0: row = max(0, row - 1)
        elif action == 1: row = min(self.size - 1, row + 1)
        elif action == 2: col = max(0, col - 1)
        elif action == 3: col = min(self.size - 1, col + 1)
        
        next_state = (row, col)
        self.state = next_state
        
        # Base reward
        if next_state == self.goal_state:
            reward, done = 10.0, True
        elif next_state in self.pitfalls:
            reward, done = -10.0, True
        else:
            reward, done = -1.0, False
        
        # Reward shaping: bonus for moving closer to goal
        old_dist = abs(old_state[0] - self.goal_state[0]) + abs(old_state[1] - self.goal_state[1])
        new_dist = abs(next_state[0] - self.goal_state[0]) + abs(next_state[1] - self.goal_state[1])
        
        if not done:
            if new_dist < old_dist:
                reward += 0.5  # Bonus for getting closer
            elif new_dist > old_dist:
                reward -= 0.5  # Penalty for moving away
        
        return next_state, reward, done, {}

# Train with shaped rewards
env_shaped = GridWorldShaped()
Q_shaped = {(r, c): np.zeros(4) for r in range(5) for c in range(5)}
epsilon = 1.0
rewards_shaped = []

for episode in range(3000):
    state = env_shaped.reset()
    total_reward = 0
    done = False
    steps = 0
    
    while not done and steps < 100:
        action = epsilon_greedy(Q_shaped[state], epsilon, 4)
        next_state, reward, done, _ = env_shaped.step(action)
        Q_shaped[state][action] += 0.1 * (reward + 0.95 * np.max(Q_shaped[next_state]) * (0 if done else 1) - Q_shaped[state][action])
        state = next_state
        total_reward += reward
        steps += 1
    
    rewards_shaped.append(total_reward)
    epsilon = max(0.01, epsilon * 0.995)

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
window = 100
ax.plot(np.convolve(rewards_history[:3000], np.ones(window)/window, mode='valid'), 
        label='Original Rewards', linewidth=2)
ax.plot(np.convolve(rewards_shaped, np.ones(window)/window, mode='valid'), 
        label='Shaped Rewards', linewidth=2)
ax.set_title('Reward Shaping: Faster Learning with Intermediate Rewards', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("🎯 Reward shaping significantly speeds up early learning!")


### Solution 9: Discretization for Continuous State Spaces

**Concept:** Many real-world problems have continuous state spaces. One simple approach is to discretize the continuous variables into bins and apply tabular Q-Learning. However, as the number of dimensions increases, this suffers from the "curse of dimensionality."

**Challenges:**
- Choosing the right number of bins is difficult
- Too few bins lose important information
- Too many bins make the Q-Table impractically large
- This is exactly why Deep RL (using neural networks) was developed!


In [ ]:
# Solution 9: Simple Continuous State Discretization
class SimpleContinuousEnv:
    """A simple 1D continuous environment: agent must reach position 5.0."""
    def __init__(self):
        self.position = 0.0
        self.target = 5.0
        self.n_bins = 10
        self.bin_edges = np.linspace(-1, 6, self.n_bins + 1)
        
    def reset(self):
        self.position = np.random.uniform(-0.5, 0.5)
        return self._discretize(self.position)
    
    def _discretize(self, position):
        """Convert continuous position to discrete bin index."""
        bin_idx = np.digitize(position, self.bin_edges) - 1
        return np.clip(bin_idx, 0, self.n_bins - 1)
    
    def step(self, action):
        """Action: 0 = move left (-0.5), 1 = move right (+0.5)"""
        move = -0.5 if action == 0 else 0.5
        self.position += move
        self.position = np.clip(self.position, -1, 6)
        
        # Reward: negative distance to target
        reward = -abs(self.position - self.target)
        done = abs(self.position - self.target) < 0.3
        
        if done:
            reward += 10  # Bonus for reaching target
        
        return self._discretize(self.position), reward, done, {}

# Train on continuous environment
env_cont = SimpleContinuousEnv()
Q_cont = {i: np.zeros(2) for i in range(env_cont.n_bins)}
epsilon = 1.0
rewards_cont = []

for episode in range(5000):
    state = env_cont.reset()
    total_reward = 0
    done = False
    steps = 0
    
    while not done and steps < 50:
        action = epsilon_greedy(Q_cont[state], epsilon, 2)
        next_state, reward, done, _ = env_cont.step(action)
        Q_cont[state][action] += 0.1 * (reward + 0.95 * np.max(Q_cont[next_state]) * (0 if done else 1) - Q_cont[state][action])
        state = next_state
        total_reward += reward
        steps += 1
    
    rewards_cont.append(total_reward)
    epsilon = max(0.01, epsilon * 0.995)

# Visualize learned policy
fig, ax = plt.subplots(figsize=(10, 5))
positions = np.linspace(-1, 6, 100)
actions = []
for pos in positions:
    bin_idx = env_cont._discretize(pos)
    actions.append(np.argmax(Q_cont[bin_idx]))

ax.plot(positions, actions, linewidth=3, color='#4ECDC4')
ax.axvline(x=5.0, color='#FF6B6B', linestyle='--', linewidth=2, label='Target')
ax.set_title('Learned Policy for Continuous Environment\n(0 = Left, 1 = Right)', fontsize=13, fontweight='bold')
ax.set_xlabel('Position')
ax.set_ylabel('Best Action')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Left', 'Right'])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("🎯 The agent learns to move right when left of target, but discretization limits precision!")


### Solution 10: SARSA Algorithm Implementation

**Concept:** SARSA is an on-policy algorithm, meaning it learns the Q-values for the policy it is currently following. Unlike Q-Learning (off-policy), which learns the optimal policy independently of the current behavior, SARSA updates using the actual next action taken.

**Key Differences:**
- **Q-Learning:** $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$
- **SARSA:** $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma Q(s',a') - Q(s,a)]$
- SARSA is generally more conservative and can perform better in stochastic environments
- Q-Learning is more aggressive in pursuing optimal values


In [ ]:
# Solution 10: SARSA Implementation
def train_sarsa(env, n_episodes=5000, alpha=0.1, gamma=0.95, 
                epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
    """Train agent using SARSA algorithm."""
    Q = {(r, c): np.zeros(4) for r in range(env.size) for c in range(env.size)}
    rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q[state], epsilon, 4)
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            next_state, reward, done, _ = env.step(action)
            
            # Choose next action (on-policy)
            next_action = epsilon_greedy(Q[next_state], epsilon, 4)
            
            # SARSA update (uses actual next action, not max)
            Q[state][action] += alpha * (reward + gamma * Q[next_state][next_action] * (0 if done else 1) - Q[state][action])
            
            state = next_state
            action = next_action
            total_reward += reward
            steps += 1
        
        rewards.append(total_reward)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    return Q, rewards

# Train both algorithms on the same environment
env_compare = GridWorld()
Q_qlearning, rewards_qlearning = train_with_params(env_compare, 0.1, 0.95, 5000)
env_compare = GridWorld()  # Fresh environment
Q_sarsa, rewards_sarsa = train_sarsa(env_compare, 5000, 0.1, 0.95)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 100
axes[0].plot(np.convolve(rewards_qlearning, np.ones(window)/window, mode='valid'), 
             label='Q-Learning', linewidth=2, color='#4ECDC4')
axes[0].plot(np.convolve(rewards_sarsa, np.ones(window)/window, mode='valid'), 
             label='SARSA', linewidth=2, color='#FF6B6B')
axes[0].set_title('Learning Curve: Q-Learning vs SARSA', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Smoothed Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare final policies
# Calculate average max Q-value per state as a proxy for policy quality
q_values = [np.max(Q_qlearning[s]) for s in Q_qlearning]
sarsa_values = [np.max(Q_sarsa[s]) for s in Q_sarsa]
axes[1].bar(['Q-Learning', 'SARSA'], [np.mean(q_values), np.mean(sarsa_values)], 
            color=['#4ECDC4', '#FF6B6B'], edgecolor='black', linewidth=1)
axes[1].set_title('Average Max Q-Value per State', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Q-Value')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("🔄 Q-Learning often converges to slightly better policies, but SARSA is more stable during learning!")


---
## 🎉 Congratulations!

You have now built your first reinforcement learning agent from scratch — a major milestone toward understanding intelligent decision-making systems. 🚀

Today you mastered:
- ✅ The fundamental concepts of Reinforcement Learning
- ✅ The Q-Learning algorithm and the Bellman equation
- ✅ The exploration vs exploitation dilemma
- ✅ Building custom environments and training loops
- ✅ Visualizing learned policies and Q-Tables
- ✅ Applying Q-Learning to classic Gym environments
- ✅ Understanding the limitations that led to Deep RL
- ✅ Comparing algorithms and tuning hyperparameters

### 🔮 Tomorrow's Teaser
**Day 73: Deep Q-Networks (DQN) and Modern Reinforcement Learning**
Tomorrow we take the next giant leap! We'll replace our Q-Table with a neural network, enabling our agents to handle massive state spaces and continuous environments. We'll implement DQN from scratch, explore experience replay, and see how DeepMind used these techniques to master Atari games. Get ready for the deep learning revolution in RL! 🤖🧠

---
*Python & AI Learning Path | Day 72 / 369* ⭐
*Keep learning, keep building, keep growing!* 💪
